# 01. Leíró Statisztika és Exploratív Adatelemzés (EDA)

**Cél**: A kőbányai lakáspiaci kínálat fundamentális leíró statisztikáinak feltárása, az eloszlások alakjának (ferdeség, csúcsosság), valamint a térbeli szóródási sajátosságoknak a vizsgálata.

**Módszertan**: Paraméteres és nem-paraméteres mutatók (átlag, medián, interkvartilis terjedelem, szórás), hisztogramok, sűrűségfüggvények (KDE) és többdimenziós eloszlásvizsgálat.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
from scipy import stats

df = load_szamitott_master()
elado = df[df['listing_type'] == 'elado'].copy()
print(f"Eladó lakások száma az EDA mintában: {len(elado)} db.")

### 1. Főbb Leíró Statisztikai Mutatók
A kőbányai eladó lakások kulcsváltozóinak központi tendenciái és szóródási mérőszámai.

In [ ]:
kpi_cards = [
    ("Átlagos Kínálati Ár", fmt_mft(elado['price_huf'].mean() / 1e6), f"Medián: {fmt_mft(elado['price_huf'].median() / 1e6)}", "#1e3a8a"),
    ("Átlagos Ár / m²", fmt_huf(elado['nm_ar_huf'].mean()), f"Medián: {fmt_huf(elado['nm_ar_huf'].median())}", "#2563eb"),
    ("Átlagos Alapterület", f"{elado['alapterulet_nm'].mean():.1f} m²", f"Medián: {elado['alapterulet_nm'].median():.1f} m²", "#059669"),
    ("Átlagos Szobaszám", f"{elado['szobaszam_osszes'].mean():.1f} szoba", f"Módusz: {elado['szobaszam_osszes'].mode()[0]} szoba", "#d97706"),
    ("Ár / m² Szórás", fmt_huf(elado['nm_ar_huf'].std()), f"Relatív szórás: {elado['nm_ar_huf'].std()/elado['nm_ar_huf'].mean()*100:.1f}%", "#dc2626"),
    ("Ár / m² Ferdeség", f"{stats.skew(elado['nm_ar_huf'].dropna()):.2f}", "Jobbra ferde (log-normális)", "#7c3aed")
]
display(HTML(kpi_grid_html(kpi_cards)))

# Részletes leíró táblázat
desc_vars = ['price_huf', 'nm_ar_huf', 'alapterulet_nm', 'szobaszam_osszes']
desc_names = ['Ár (HUF)', 'Ár / m² (HUF)', 'Alapterület (m²)', 'Szobaszám']
stat_df = elado[desc_vars].describe().T
stat_df['skew'] = [stats.skew(elado[c].dropna()) for c in desc_vars]
stat_df['kurtosis'] = [stats.kurtosis(elado[c].dropna()) for c in desc_vars]
stat_df.index = desc_names
stat_df.columns = ['Darab', 'Átlag', 'Szórás', 'Min', '25% (Q1)', 'Medián (Q2)', '75% (Q3)', 'Max', 'Ferdeség', 'Csúcsosság']

html_table = "<div style='overflow-x:auto; margin: 15px 0;'>" + stat_df.round(2).to_html(classes='table table-bordered table-hover') + "</div>"
display(HTML(html_table))

### 2. Négyzetméterárak Eloszlása és Log-Transzformáció
A nyers négyzetméterárak jobbra ferde eloszlást mutatnak, míg a logaritmikus transzformáció közel normális eloszlást eredményez, ami elengedhetetlen a lineáris regressziós modellekhez.

In [ ]:
fig1 = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Nyers Négyzetméterár Eloszlása (Ft/m²)', 'Logaritmikus Négyzetméterár Eloszlása (ln(Ár/m²))')
)

fig1.add_trace(
    go.Histogram(x=elado['nm_ar_huf'], nbinsx=35, name='Ár/m²', marker_color='#2563eb', opacity=0.8),
    row=1, col=1
)

fig1.add_trace(
    go.Histogram(x=np.log(elado['nm_ar_huf']), nbinsx=35, name='ln(Ár/m²)', marker_color='#059669', opacity=0.8),
    row=1, col=2
)

fig1.update_layout(
    title_text='Fajlagos árak eloszlása és normálissá alakítása log-transzformációval',
    template=PLOTLY_TEMPLATE,
    height=450,
    showlegend=False
)
fig1.show()

### 3. Városrészi Összehasonlítás és Árszintek
A négyzetméterárak és lakásméretek variabilitása Kőbánya egyes kerületrészeiben.

In [ ]:
# Városrészi boxplot
fig2 = px.box(
    elado,
    x='varosresz',
    y='nm_ar_huf',
    color='varosresz',
    title='Négyzetméterár dobozábra (Boxplot) városrészenként',
    labels={'varosresz': 'Városrész', 'nm_ar_huf': 'Ár / m² (HUF)'},
    template=PLOTLY_TEMPLATE
)
fig2.update_layout(xaxis_tickangle=-30, height=450, showlegend=False)
fig2.show()

# Méret vs Ár szórásdiagram OLS trenddel
fig3 = px.scatter(
    elado,
    x='alapterulet_nm',
    y='ar_millio_ft',
    color='varosresz',
    trendline='ols',
    title='Alapterület vs. Kínálati Ár összefüggés városrészenként',
    labels={'alapterulet_nm': 'Alapterület (m²)', 'ar_millio_ft': 'Ár (Millió Ft)', 'varosresz': 'Városrész'},
    template=PLOTLY_TEMPLATE
)
fig3.update_layout(height=480)
fig3.show()

### 4. Interaktív EDA Vizualizációs Panel
A JupyterLab környezetben a lenti vezérlőkkel interaktívan elemezhető bármely kiválasztott változó eloszlása a kiválasztott városrészekre szűrve.

In [ ]:
w_var = widgets.Dropdown(
    options=[('Négyzetméterár (HUF)', 'nm_ar_huf'), ('Kínálati Ár (M Ft)', 'ar_millio_ft'), ('Alapterület (m²)', 'alapterulet_nm'), ('Szobaszám', 'szobaszam_osszes')],
    value='nm_ar_huf',
    description='Változó:'
)
w_varos = widgets.SelectMultiple(
    options=VAROSRESZEK,
    value=tuple(VAROSRESZEK[:4]),
    description='Városrészek:'
)
out_eda = widgets.Output()

def frissit_eda(*args):
    sub = elado[elado['varosresz'].isin(w_varos.value)]
    with out_eda:
        clear_output(wait=True)
        col = w_var.value
        fig = px.histogram(
            sub, x=col, color='varosresz', barmode='overlay',
            title=f'{w_var.label} eloszlása a kiválasztott városrészekben (N={len(sub)})',
            template=PLOTLY_TEMPLATE, opacity=0.7
        )
        fig.show()

w_var.observe(frissit_eda, names='value')
w_varos.observe(frissit_eda, names='value')

display(widgets.HBox([w_var, w_varos]))
display(out_eda)
frissit_eda()